# Topic 1: User-Defined Functions (UDFs)

> **Phase 3 → Week 5 → Topic 1**
>
> Prerequisites: Week 4 complete (Joins, Window Functions, Built-in Functions)

---

## Why UDFs Exist — and Why They're a Last Resort

Built-in functions cover ~90% of needs. When you need custom logic, you have two options:

```
Option 1: Python UDF                    Option 2: Pandas UDF (Vectorized)
─────────────────────────────────────   ────────────────────────────────────────
JVM → serialize row to Python           JVM → send Arrow BATCH to Python
     → run Python function              → run Pandas/NumPy on entire batch
     → serialize result back to JVM     → return Arrow batch to JVM

Cost per row: HIGH (row-by-row)         Cost: one round-trip per partition
Catalyst optimization: NONE             Catalyst optimization: NONE
Speed vs built-in: 10-100x slower       Speed: 3-5x slower than built-in
                                         (but much faster than Python UDF)
```

**UDF performance order (fastest → slowest):**
1. Built-in `pyspark.sql.functions` — always prefer
2. Pandas UDF (vectorized) — use for complex logic or when you need NumPy/Pandas
3. Python UDF — use only when 1 and 2 aren't possible

---

## Interview Cheat Sheet

**Q: What is a UDF in Spark? What are its downsides?**
> A UDF (User-Defined Function) is a custom function registered with Spark to extend the DataFrame API. Downsides: (1) Catalyst cannot optimize inside a UDF — it's a black box; (2) Each row is serialized from JVM to Python and back — row-by-row overhead; (3) Null handling must be done manually inside the function.

**Q: What is a Pandas UDF and how is it faster than a regular UDF?**
> A Pandas UDF (aka vectorized UDF) uses Apache Arrow to transfer an entire batch/partition of data from JVM to Python as a Pandas Series at once — not row-by-row. The Python function receives a Pandas Series and returns one. This eliminates most serialization overhead. Still slower than built-in functions but 3-10x faster than regular Python UDFs.

**Q: What happens when a Python UDF receives a null value?**
> By default, if the input is `null`, Spark passes `None` to your Python function. If your function raises an exception on `None`, the job fails. You must handle `None` explicitly inside the UDF, or add a guard: `if val is None: return None`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import udf, pandas_udf
import pandas as pd

spark = SparkSession.builder \
    .appName("Week5 - UDFs") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)
orders    = spark.read.csv("/workspace/week4/data/orders.csv",    header=True, inferSchema=True)
print("Ready")

## Part 1: Python UDF — The Basics

In [ ]:
# Example: classify order amounts into tiers
# Built-in when() can do this — this is just for demonstration

# Method 1: @udf decorator
@udf(returnType=StringType())
def classify_amount(amount):
    if amount is None:
        return "Unknown"          # always handle None!
    elif amount >= 1000:
        return "High"
    elif amount >= 200:
        return "Medium"
    elif amount >= 50:
        return "Low"
    else:
        return "Micro"

# Method 2: udf() function
def title_case(s):
    if s is None:
        return None
    return s.title()

title_case_udf = udf(title_case, StringType())

# Apply UDFs
result = orders.withColumn("amount_tier", classify_amount(F.col("amount"))) \
               .withColumn("status_title", title_case_udf(F.col("status")))

print("UDF results:")
result.select("order_id", "amount", "amount_tier", "status", "status_title").show()

In [ ]:
# Register UDF for use in SQL queries
spark.udf.register("classify_amount_sql", classify_amount)
orders.createOrReplaceTempView("orders_view")

spark.sql("""
    SELECT order_id, amount, classify_amount_sql(amount) AS tier
    FROM orders_view
    ORDER BY amount DESC
    LIMIT 5
""").show()

In [ ]:
# UDF with complex return type — StructType
@udf(returnType=StructType([
    StructField("first", StringType(), True),
    StructField("last",  StringType(), True),
    StructField("length", IntegerType(), True),
]))
def parse_name(full_name):
    if full_name is None:
        return None
    parts = full_name.strip().split(" ")
    first = parts[0]
    last  = parts[-1] if len(parts) > 1 else None
    return (first, last, len(full_name))

parsed = customers.withColumn("parsed_name", parse_name(F.col("name")))
print("UDF returning StructType:")
parsed.select(
    "name",
    "parsed_name.first",
    "parsed_name.last",
    "parsed_name.length"
).show()

In [ ]:
# UDF with ArrayType return type
@udf(returnType=ArrayType(StringType()))
def tokenize(text):
    if text is None:
        return []
    return [word.lower().strip() for word in text.split()]

text_df = spark.createDataFrame([
    (1, "Apache Spark is amazing"),
    (2, "PySpark makes big data easy"),
    (3, None),
], ["id", "text"])

text_df.withColumn("tokens", tokenize(F.col("text"))) \
       .withColumn("word_count", F.size("tokens")) \
       .show(truncate=False)

## Part 2: Pandas UDF (Vectorized) — The Fast Way

In [ ]:
# Pandas UDF — function receives a pd.Series, returns a pd.Series
# Enable Arrow-based columnar transfer
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

@pandas_udf(StringType())
def classify_amount_pandas(amounts: pd.Series) -> pd.Series:
    # Operates on entire partition at once — no row-by-row Python loop
    return pd.cut(
        amounts,
        bins=[-float('inf'), 50, 200, 1000, float('inf')],
        labels=["Micro", "Low", "Medium", "High"]
    ).astype(str)

print("Pandas UDF (vectorized):")
orders.withColumn("tier_pandas", classify_amount_pandas(F.col("amount"))) \
      .select("order_id", "amount", "tier_pandas") \
      .show()

In [ ]:
# Pandas UDF for date arithmetic (complex logic)
import pandas as pd
import numpy as np

@pandas_udf(DoubleType())
def days_until_next_quarter(dates: pd.Series) -> pd.Series:
    """Calculate days until end of current quarter."""
    dates = pd.to_datetime(dates)
    quarter_end = dates.dt.to_period('Q').dt.to_timestamp('Q') + pd.offsets.MonthEnd(0)
    return (quarter_end - dates).dt.days.astype(float)

orders.withColumn(
    "days_to_quarter_end", days_until_next_quarter(F.col("order_date"))
).select("order_id", "order_date", "days_to_quarter_end").show()

In [ ]:
# Pandas UDF with multiple columns (use struct trick)
@pandas_udf(DoubleType())
def weighted_score(amount: pd.Series, quantity: pd.Series) -> pd.Series:
    # Weight amount 70%, quantity contribution 30%
    max_amount = amount.max() if amount.max() > 0 else 1
    max_qty    = quantity.max() if quantity.max() > 0 else 1
    return (0.7 * amount / max_amount + 0.3 * quantity / max_qty) * 100

orders.withColumn(
    "score", F.round(weighted_score(F.col("amount"), F.col("quantity")), 1)
).select("order_id", "amount", "quantity", "score") \
 .orderBy(F.col("score").desc()) \
 .show()

## Part 3: Performance Comparison

In [ ]:
import time

# Large dataset for fair timing
large_df = spark.range(100_000).withColumn(
    "amount", (F.rand() * 2000).cast("double")
)

# Option 1: Built-in when()
t0 = time.time()
large_df.withColumn("tier",
    F.when(F.col("amount") >= 1000, "High")
     .when(F.col("amount") >= 200, "Medium")
     .when(F.col("amount") >= 50, "Low")
     .otherwise("Micro")
).count()
builtin_time = time.time() - t0

# Option 2: Python UDF
t0 = time.time()
large_df.withColumn("tier", classify_amount(F.col("amount"))).count()
udf_time = time.time() - t0

# Option 3: Pandas UDF
t0 = time.time()
large_df.withColumn("tier", classify_amount_pandas(F.col("amount"))).count()
pandas_udf_time = time.time() - t0

print(f"Performance on 100k rows:")
print(f"  Built-in when():  {builtin_time:.3f}s")
print(f"  Pandas UDF:       {pandas_udf_time:.3f}s  ({pandas_udf_time/builtin_time:.1f}x slower than built-in)")
print(f"  Python UDF:       {udf_time:.3f}s  ({udf_time/builtin_time:.1f}x slower than built-in)")

## Part 4: When to Use Each

| Scenario | Best Choice |
|----------|------------|
| String manipulation (upper, concat, split) | Built-in `F.upper()`, `F.split()` |
| Date math (datediff, date_format) | Built-in date functions |
| Conditional logic (IF/ELSE) | Built-in `F.when().otherwise()` |
| Mathematical transforms | Built-in `F.sqrt()`, `F.pow()` etc. |
| Complex business logic (regex, scoring) | Pandas UDF (vectorized) |
| ML feature engineering (requires NumPy) | Pandas UDF |
| Legacy Python function, can't vectorize | Python UDF (last resort) |
| Calling external API per row | Python UDF (no choice) |

In [ ]:
# Common mistake: using UDF when built-in exists

# BAD — Python UDF for something built-in can do
@udf(StringType())
def bad_upper(s):
    return s.upper() if s else None

# GOOD — use built-in
df = customers.select(
    F.upper("name").alias("good_way"),    # use this
    bad_upper("name").alias("bad_way")    # never do this for simple ops
)
print("Result is same, but good_way is 10-100x faster:")
df.show(5)

## Exercises

1. Write a Python UDF that takes a `country` name and returns its continent (`Asia`, `Europe`, `America`, `Africa`, `Other`). Apply it to the customers table.
2. Rewrite the UDF from Exercise 1 as a Pandas UDF.
3. Write a Pandas UDF that normalizes `amount` values to a 0-100 scale (min-max normalization) using NumPy.
4. When would you use a Python UDF over a Pandas UDF? Give 2 scenarios.
5. A colleague wrote this: `@udf(DoubleType()) def avg_udf(values): return sum(values)/len(values)` and applied it to a list column. What's wrong? How would you fix it?

In [ ]:
# Exercise 1: Country → Continent UDF
@udf(StringType())
def get_continent(country):
    if country is None:
        return "Unknown"
    mapping = {
        "India": "Asia", "China": "Asia", "Japan": "Asia",
        "South Korea": "Asia",
        "Germany": "Europe", "Italy": "Europe", "UK": "Europe",
        "USA": "America",
        "Nigeria": "Africa",
    }
    return mapping.get(country, "Other")

customers.withColumn("continent", get_continent(F.col("country"))) \
         .select("name", "country", "continent") \
         .show()

In [ ]:
# Exercise 3: Min-max normalization Pandas UDF
import numpy as np

@pandas_udf(DoubleType())
def normalize_amount(amounts: pd.Series) -> pd.Series:
    min_val = amounts.min()
    max_val = amounts.max()
    if max_val == min_val:
        return pd.Series([0.0] * len(amounts))
    return ((amounts - min_val) / (max_val - min_val) * 100).round(2)

orders.withColumn("normalized_amount", normalize_amount(F.col("amount"))) \
      .select("order_id", "amount", "normalized_amount") \
      .show()